# EvoX-FS — run one dataset

This notebook only **calls** the code in `src/evoxfs/`. To change a setting, edit `configs/base.yaml` (all datasets) or `configs/<dataset>.yaml` (one dataset), not this notebook.

1. Set `DATASET` below (`wine`, `breast_cancer`, `sonar`, `ionosphere`).
2. Runtime → Run all.
3. Results (CSV + figures) are saved to Google Drive → `MyDrive/EvoXFS_results`. If the session disconnects, run again: finished runs are reused from the checkpoints.

In [ ]:
DATASET = "wine"          # wine | breast_cancer | sonar | ionosphere
REPO_BRANCH = "main"       # which version of the code to use (a branch or a tag, e.g. "v1.0")

## 1. Setup (Colab): Drive + code

In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !rm -rf /content/evoxfs-code1
    !git clone -q --depth 1 --branch {REPO_BRANCH} https://github.com/aydafiruzy/evoxfs-code1 /content/evoxfs-code1
    !pip install -q -e /content/evoxfs-code1
    sys.path.insert(0, "/content/evoxfs-code1/src")

!git -C /content/evoxfs-code1 log -1 --format='Code version: %h  %s' 2>/dev/null || true

## 2. Configuration, data, SHAP

In [ ]:
from IPython.display import display
from evoxfs import load_config, load_data, compute_shap
from evoxfs.config import print_summary
from evoxfs.data import describe
from evoxfs.shap_tools import shap_table, plot_shap

cfg = load_config(DATASET)
print_summary(cfg)
data = load_data(cfg)
display(describe(data, cfg["test_size"]))

shap_res = compute_shap(data, cfg)
display(shap_table(shap_res, data, cfg).round(4))
plot_shap(shap_res, data, cfg)

## 3. Hyperparameter tuning (Default vs Greedy)

In [ ]:
from evoxfs.tuning import run_tuning
from evoxfs.fitness import Context

tuning_df, tuned = run_tuning(data, cfg)
display(tuning_df.drop(columns="params").round(4))
ctx = Context(cfg, data, shap_res.norm,
              eval_params={m: tuned[m] for m in cfg["eval_models"]},
              fitness_params=tuned["Fitness XGBoost"])

## 4. Main experiment: 8 methods × populations × runs (long — checkpointed)

In [ ]:
from evoxfs.experiment import run_main, METHODS
display(__import__("pandas").DataFrame(METHODS).T)
experiment_df, masks, histories = run_main(ctx)

## 5. Results and statistics

In [ ]:
from evoxfs.analysis import Results
res = Results(ctx, experiment_df, masks, histories)
res.convergence()
res.summaries()
res.statistics()

## 6. Additional experiments: control fitness (no SHAP term) + sensitivity of w_size

In [ ]:
from evoxfs.experiment import run_extra
extra_df = run_extra(ctx)
res.extra(extra_df)

## 7. Save every table to Drive

In [ ]:
res.save(tuning_df)